# Forklift Anomaly Detection Skeleton


## 0. パッケージのインストール


In [ ]:
#%pip install jupyterlab==4.4.1 matplotlib==3.10.1 numpy==2.2.4 pandas==2.2.3 scipy==1.15.2 scikit-learn==1.6.1 librosa==0.10.2.post1 soundfile==0.13.1 opencv-python-headless==4.11.0.86


In [ ]:
from __future__ import annotations

from io import BytesIO
from pathlib import Path
from typing import Any

import shutil
import subprocess

import cv2
import librosa
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image, display
from scipy import signal
from scipy.spatial import Delaunay
from sklearn.decomposition import PCA
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler


In [ ]:
DATA_DIR = Path("../data")
NORMAL_VIDEO_DIR = DATA_DIR / "normal"
ANOMARY_VIDEO_DIR = DATA_DIR / "anomary"
REFERENCE_VIDEO_PATH = DATA_DIR / "sample.mp4"
OUTPUT_DIR = Path("../outputs")
ANOMARY_LIKE_NORMAL_LIST_PATH = DATA_DIR / "anomary_like_normal.txt"

SPLIT_BASE_DIR = OUTPUT_DIR / "audio_dataset_split"
TRAIN_NORMAL_DIR = SPLIT_BASE_DIR / "train" / "normal"
EVAL_NORMAL_DIR = SPLIT_BASE_DIR / "eval" / "normal"
EVAL_ANOMARY_DIR = SPLIT_BASE_DIR / "eval" / "anomary"
AUDIO_CACHE_DIR = OUTPUT_DIR / "audio_cache"
EVALUATION_RESULTS_PATH = OUTPUT_DIR / "audio_evaluation_results.csv"
SPLIT_MANIFEST_PATH = SPLIT_BASE_DIR / "split_manifest.csv"

VIDEO_PROCESSING_ENABLED = True
VIDEO_SAMPLE_FPS = 2
VIDEO_MAX_SAMPLE_FRAMES = 200
VIDEO_LETTERBOX_BLACK_THRESHOLD = 8
VIDEO_UNIFORM_DOMINANT_RATIO_THRESHOLD = 0.95
VIDEO_UNIFORM_COLOR_TOLERANCE = 30.0
VEHICLE_SCORE_HIST_BINS = 64
VEHICLE_SCORE_QUANTILES = [0.25, 0.5, 0.75]
VEHICLE_LOW_SEGMENT_MAX = 1
VEHICLE_COMPONENT_MIN_AREA_RATIO = 0.002
VEHICLE_COMPONENT_KERNEL_SIZE = 5
EDGE_CANNY_LOW_THRESHOLD = 60
EDGE_CANNY_HIGH_THRESHOLD = 160
EDGE_PERSISTENCE_THRESHOLD = 0.4
EDGE_PREVIEW_FRAME_COUNT = 5
EDGE_PERSISTENCE_WEAK_THRESHOLD_RATIO = 0.6
EDGE_HYSTERESIS_KERNEL_SIZE = 5
EDGE_HYSTERESIS_MAX_ITERATIONS = 128
EDGE_ORIENTATION_TOLERANCE_DEG = 25.0
EDGE_COHERENCE_KERNEL_SIZE = 3
EDGE_COHERENCE_EPS = 1e-6
EDGE_COHERENCE_MIN_PERSISTENCE = 0.1
EDGE_COHERENCE_HIGH_QUANTILE = 0.8
EDGE_LOW_COHERENCE_QUANTILE = 0.2
EDGE_LOW_COHERENCE_REGION_RATIO_THRESHOLD = 0.4
EDGE_COHERENCE_MIN_COMPONENT_AREA_RATIO = 0.0005
EDGE_COMBINED_SCORE_TARGET_QUANTILE = 0.8
EDGE_DISPLAY_CONNECTED_MIN_PERSISTENCE_RATIO = 0.5
EDGE_DISPLAY_CONNECTED_COHERENCE_QUANTILE = 0.7
EDGE_DISPLAY_CONNECT_KERNEL_SIZE = 3
EDGE_DISPLAY_CONNECT_ITERATIONS = 1
EDGE_DISPLAY_BORDER_MAX_DISTANCE = 60.0
EDGE_DISPLAY_BORDER_DIRECTION_RADIUS = 12
EDGE_DISPLAY_BORDER_DIRECTION_NEIGHBORS = 6
EDGE_CONCAVE_HULL_ALPHA_RADIUS = 12.0
EDGE_CONCAVE_HULL_POINT_STEP = 2
EDGE_FINAL_CONNECT_KERNEL_SIZE = 9
EDGE_FINAL_CONNECT_ITERATIONS = 1
EDGE_FINAL_MIN_RECT_AREA_PIXELS = 518
EDGE_RAW_CONNECT_KERNEL_SIZE = 3
EDGE_RAW_CONNECT_ITERATIONS = 1
VEHICLE_GROWTH_SEED_SEGMENT_MAX = 0
VEHICLE_GROWTH_CANDIDATE_SEGMENT_MAX = 1
VEHICLE_GROWTH_BARRIER_KERNEL_SIZE = 3
VEHICLE_GROWTH_KERNEL_SIZE = 5
VEHICLE_GROWTH_REFINEMENT_KERNEL_SIZE = 5
VEHICLE_GROWTH_MAX_ITERATIONS = 128
VEHICLE_GROWTH_MIN_AREA_RATIO = 0.002

TRAIN_SPLIT_RATIO = 0.8
SPLIT_RANDOM_SEED = 42

AUDIO_SAMPLE_RATE = 22050
AUDIO_MONO_CHANNELS = 1
AUDIO_HIGHPASS_CUTOFF_HZ = 20.0
AUDIO_N_FFT = 1024
AUDIO_WIN_LENGTH = 1024
AUDIO_HOP_LENGTH = 256
AUDIO_N_MELS = 64
AUDIO_FMIN_HZ = 20.0
AUDIO_FMAX_HZ = 10000.0
AUDIO_WINDOW_SEC = 0.5
AUDIO_WINDOW_HOP_SEC = 0.25
AUDIO_BAND_LIMITS_HZ = [(20, 200), (200, 1000), (1000, 4000), (4000, 10000)]
AUDIO_PCA_COMPONENTS = 32
AUDIO_ISOLATION_FOREST_RANDOM_STATE = 42
AUDIO_CLIP_SCORE_TOP_K = 3
NORMAL_SCORE_THRESHOLD_PERCENTILE = 95.0


## 1. 動画の読み込み

In [ ]:
def resolve_reference_video_path() -> Path | None:
    if REFERENCE_VIDEO_PATH.exists():
        return REFERENCE_VIDEO_PATH

    normal_candidates = sorted(path for path in NORMAL_VIDEO_DIR.glob("*.mp4") if path.is_file())
    if normal_candidates:
        return normal_candidates[0]
    return None


def read_video_metadata(video_path: Path) -> dict[str, Any]:
    capture = cv2.VideoCapture(str(video_path))
    if not capture.isOpened():
        raise FileNotFoundError(f"Failed to open video: {video_path}")

    metadata = {
        "video_path": str(video_path),
        "frame_count": int(capture.get(cv2.CAP_PROP_FRAME_COUNT)),
        "fps": float(capture.get(cv2.CAP_PROP_FPS)),
        "width": int(capture.get(cv2.CAP_PROP_FRAME_WIDTH)),
        "height": int(capture.get(cv2.CAP_PROP_FRAME_HEIGHT)),
    }
    metadata["duration_sec"] = (
        metadata["frame_count"] / metadata["fps"] if metadata["fps"] > 0 else np.nan
    )
    capture.release()
    return metadata


def iter_sampled_frames(
    video_path: Path,
    sampling_fps: int = VIDEO_SAMPLE_FPS,
    max_frames: int | None = VIDEO_MAX_SAMPLE_FRAMES,
) -> list[np.ndarray]:
    metadata = read_video_metadata(video_path)
    native_fps = metadata["fps"] if metadata["fps"] > 0 else 1.0
    step = max(int(round(native_fps / max(sampling_fps, 1))), 1)

    frames: list[np.ndarray] = []
    capture = cv2.VideoCapture(str(video_path))
    frame_index = 0
    while capture.isOpened():
        success, frame_bgr = capture.read()
        if not success:
            break

        if frame_index % step == 0:
            frames.append(cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB))
            if max_frames is not None and len(frames) >= max_frames:
                break
        frame_index += 1

    capture.release()
    return frames


def show_image(image: np.ndarray, title: str, figsize: tuple[int, int] = (8, 4)) -> None:
    figure, ax = plt.subplots(figsize=figsize)
    ax.imshow(image)
    ax.set_title(title)
    ax.axis("off")
    buffer = BytesIO()
    figure.savefig(buffer, format="png", bbox_inches="tight")
    plt.close(figure)
    buffer.seek(0)
    display(Image(data=buffer.getvalue()))


def show_video_figure(figure: plt.Figure) -> None:
    buffer = BytesIO()
    figure.savefig(buffer, format="png", bbox_inches="tight")
    plt.close(figure)
    buffer.seek(0)
    display(Image(data=buffer.getvalue()))


reference_video_path = resolve_reference_video_path()
reference_metadata = None
reference_sampled_stacked_frames: list[np.ndarray] = []
reference_stacked_frame = None

if reference_video_path is None:
    print("No reference video is available for vehicle-region segmentation.")
else:
    print("reference video:", reference_video_path)
    reference_metadata = read_video_metadata(reference_video_path)
    print("video metadata:")
    print(reference_metadata)

    reference_sampled_stacked_frames = iter_sampled_frames(reference_video_path)
    print("sampled stacked frame count:", len(reference_sampled_stacked_frames))

    if reference_sampled_stacked_frames:
        reference_stacked_frame = reference_sampled_stacked_frames[0]
        print("first stacked frame shape:", reference_stacked_frame.shape)
        show_image(reference_stacked_frame, title="Loaded First Stacked Frame")
    else:
        print("No sampled stacked frames could be loaded from the reference video.")


## 1.5 暫定: 左右レターボックス除去

In [ ]:
def detect_side_letterbox_bounds(
    stacked_frame: np.ndarray,
    black_threshold: int = VIDEO_LETTERBOX_BLACK_THRESHOLD,
    min_non_black_ratio: float = 0.01,
) -> tuple[int, int, dict[str, int]]:
    gray = cv2.cvtColor(stacked_frame, cv2.COLOR_RGB2GRAY)
    non_black_ratio = (gray > black_threshold).mean(axis=0)
    valid_columns = np.where(non_black_ratio > min_non_black_ratio)[0]

    if valid_columns.size == 0:
        crop_left = 0
        crop_right = stacked_frame.shape[1]
    else:
        crop_left = int(valid_columns[0])
        crop_right = int(valid_columns[-1]) + 1

    if crop_right <= crop_left:
        crop_left = 0
        crop_right = stacked_frame.shape[1]

    return crop_left, crop_right, {
        "original_width": int(stacked_frame.shape[1]),
        "crop_left": int(crop_left),
        "crop_right": int(crop_right),
        "cropped_width": int(crop_right - crop_left),
        "removed_left_px": int(crop_left),
        "removed_right_px": int(stacked_frame.shape[1] - crop_right),
    }


def apply_side_crop(stacked_frame: np.ndarray, crop_left: int, crop_right: int) -> np.ndarray:
    return stacked_frame[:, crop_left:crop_right, :]


def choose_common_side_letterbox_crop(stacked_frames: list[np.ndarray]) -> dict[str, int]:
    if not stacked_frames:
        return {
            "frame_count": 0,
            "original_width": 0,
            "crop_left": 0,
            "crop_right": 0,
            "cropped_width": 0,
            "removed_left_px": 0,
            "removed_right_px": 0,
        }

    crop_lefts: list[int] = []
    crop_rights: list[int] = []
    original_width = int(stacked_frames[0].shape[1])
    for stacked_frame in stacked_frames:
        crop_left, crop_right, _ = detect_side_letterbox_bounds(stacked_frame)
        crop_lefts.append(crop_left)
        crop_rights.append(crop_right)

    crop_left = int(np.median(crop_lefts))
    crop_right = int(np.median(crop_rights))
    crop_left = max(0, min(crop_left, original_width - 1))
    crop_right = max(crop_left + 1, min(crop_right, original_width))

    return {
        "frame_count": int(len(stacked_frames)),
        "original_width": original_width,
        "crop_left": int(crop_left),
        "crop_right": int(crop_right),
        "cropped_width": int(crop_right - crop_left),
        "removed_left_px": int(crop_left),
        "removed_right_px": int(original_width - crop_right),
    }


def preprocess_stacked_frames_remove_letterbox(
    stacked_frames: list[np.ndarray],
) -> tuple[list[np.ndarray], dict[str, int]]:
    if not stacked_frames:
        return [], choose_common_side_letterbox_crop(stacked_frames)

    crop_info = choose_common_side_letterbox_crop(stacked_frames)
    cropped_frames = [
        apply_side_crop(stacked_frame, crop_info["crop_left"], crop_info["crop_right"])
        for stacked_frame in stacked_frames
    ]
    return cropped_frames, crop_info


reference_preprocessed_stacked_frames: list[np.ndarray] = []
reference_preprocessed_frame = None
reference_letterbox_crop_info = None

if not reference_sampled_stacked_frames:
    print("Skip temporary letterbox removal because no sampled frames are available.")
else:
    reference_preprocessed_stacked_frames, reference_letterbox_crop_info = preprocess_stacked_frames_remove_letterbox(
        reference_sampled_stacked_frames
    )
    reference_preprocessed_frame = reference_preprocessed_stacked_frames[0]
    print("temporary common letterbox crop info:")
    print(reference_letterbox_crop_info)
    show_image(reference_stacked_frame, title="Before Temporary Letterbox Removal")
    show_image(reference_preprocessed_frame, title="After Temporary Letterbox Removal")


## 2. 前後動画分割

In [ ]:
def split_front_rear_frame(stacked_frame: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    height = stacked_frame.shape[0]
    midpoint = height // 2
    return stacked_frame[:midpoint, :, :], stacked_frame[midpoint:, :, :]


def split_front_rear_frames(stacked_frames: list[np.ndarray]) -> tuple[list[np.ndarray], list[np.ndarray]]:
    front_frames: list[np.ndarray] = []
    rear_frames: list[np.ndarray] = []
    for stacked_frame in stacked_frames:
        front_frame, rear_frame = split_front_rear_frame(stacked_frame)
        front_frames.append(front_frame)
        rear_frames.append(rear_frame)
    return front_frames, rear_frames


reference_front_frames: list[np.ndarray] = []
reference_rear_frames: list[np.ndarray] = []
reference_front_frame = None
reference_rear_frame = None

if not reference_preprocessed_stacked_frames:
    print("Skip split preview because no preprocessed frames are available.")
else:
    reference_front_frames, reference_rear_frames = split_front_rear_frames(reference_preprocessed_stacked_frames)
    print("split frame pairs:", len(reference_front_frames))
    if reference_front_frames and reference_rear_frames:
        reference_front_frame = reference_front_frames[0]
        reference_rear_frame = reference_rear_frames[0]
        print("front frame shape:", reference_front_frame.shape)
        print("rear frame shape:", reference_rear_frame.shape)
        show_image(reference_front_frame, title="Front First Frame Before Uniform-Frame Trim")
        show_image(reference_rear_frame, title="Rear First Frame Before Uniform-Frame Trim")


## 2.5 ほぼ一色のフレーム除去（前後同期）

In [ ]:
def is_nearly_uniform_frame(
    frame: np.ndarray,
    dominant_ratio_threshold: float = VIDEO_UNIFORM_DOMINANT_RATIO_THRESHOLD,
    color_tolerance: float = VIDEO_UNIFORM_COLOR_TOLERANCE,
) -> bool:
    pixels = frame.reshape(-1, 3).astype(np.float32)
    representative_color = np.median(pixels, axis=0)
    pixel_deviation = np.abs(pixels - representative_color).max(axis=1)
    near_representative_ratio = float((pixel_deviation <= color_tolerance).mean())
    return near_representative_ratio >= dominant_ratio_threshold


def count_edge_uniform_frames(frames: list[np.ndarray], reverse: bool = False) -> int:
    ordered_frames = list(reversed(frames)) if reverse else frames
    count = 0
    for frame in ordered_frames:
        if is_nearly_uniform_frame(frame):
            count += 1
        else:
            break
    return count


def trim_synchronized_uniform_frames(
    front_frames: list[np.ndarray],
    rear_frames: list[np.ndarray],
) -> tuple[list[np.ndarray], list[np.ndarray], dict[str, int]]:
    if len(front_frames) != len(rear_frames):
        raise ValueError("Front and rear frame counts must match before uniform-frame trimming.")

    total_frames = len(front_frames)
    if total_frames == 0:
        return [], [], {
            "input_frame_pairs": 0,
            "removed_from_start": 0,
            "removed_from_end": 0,
            "remaining_frame_pairs": 0,
        }

    front_uniform_start = count_edge_uniform_frames(front_frames, reverse=False)
    rear_uniform_start = count_edge_uniform_frames(rear_frames, reverse=False)
    front_uniform_end = count_edge_uniform_frames(front_frames, reverse=True)
    rear_uniform_end = count_edge_uniform_frames(rear_frames, reverse=True)

    removed_from_start = max(front_uniform_start, rear_uniform_start)
    removed_from_end = max(front_uniform_end, rear_uniform_end)

    if removed_from_start + removed_from_end >= total_frames:
        trimmed_front_frames = []
        trimmed_rear_frames = []
    else:
        end_index = total_frames - removed_from_end if removed_from_end > 0 else total_frames
        trimmed_front_frames = front_frames[removed_from_start:end_index]
        trimmed_rear_frames = rear_frames[removed_from_start:end_index]

    return trimmed_front_frames, trimmed_rear_frames, {
        "input_frame_pairs": int(total_frames),
        "front_uniform_frames_at_start": int(front_uniform_start),
        "rear_uniform_frames_at_start": int(rear_uniform_start),
        "front_uniform_frames_at_end": int(front_uniform_end),
        "rear_uniform_frames_at_end": int(rear_uniform_end),
        "removed_from_start": int(removed_from_start),
        "removed_from_end": int(removed_from_end),
        "remaining_frame_pairs": int(len(trimmed_front_frames)),
    }


reference_uniform_trim_info = None

if not reference_front_frames or not reference_rear_frames:
    print("Skip synchronized uniform-frame trim because split frames are not available.")
else:
    reference_front_frames, reference_rear_frames, reference_uniform_trim_info = trim_synchronized_uniform_frames(
        reference_front_frames,
        reference_rear_frames,
    )
    print("uniform frame trim info:")
    print(reference_uniform_trim_info)
    if reference_front_frames and reference_rear_frames:
        reference_front_frame = reference_front_frames[0]
        reference_rear_frame = reference_rear_frames[0]
        show_image(reference_front_frame, title="Front First Frame After Uniform-Frame Trim")
        show_image(reference_rear_frame, title="Rear First Frame After Uniform-Frame Trim")


## 3. 全動画の前後分割保存

In [ ]:
MOVIE_PREPROCESS_OUTPUT_DIR = OUTPUT_DIR / "movie_preprocess"
MOVIE_PREPROCESS_MANIFEST_PATH = MOVIE_PREPROCESS_OUTPUT_DIR / "movie_preprocess_manifest.csv"
MOVIE_PREPROCESS_FOURCC = "mp4v"
MOVIE_PREPROCESS_OVERWRITE = False


def find_movie_preprocess_targets() -> list[dict[str, Any]]:
    targets: list[dict[str, Any]] = []
    for category, video_dir in [
        ("normal", NORMAL_VIDEO_DIR),
        ("anomary", ANOMARY_VIDEO_DIR),
    ]:
        for video_path in sorted(video_dir.glob("*.mp4")):
            if video_path.is_file():
                targets.append({"category": category, "video_path": video_path})
    return targets


def estimate_full_video_uniform_trim(
    video_path: Path,
    crop_info: dict[str, int],
) -> dict[str, int]:
    capture = cv2.VideoCapture(str(video_path))
    if not capture.isOpened():
        raise FileNotFoundError(f"Failed to open video: {video_path}")

    total_frame_pairs = 0
    front_uniform_start = 0
    rear_uniform_start = 0
    front_uniform_end = 0
    rear_uniform_end = 0
    front_start_open = True
    rear_start_open = True

    try:
        while capture.isOpened():
            success, stacked_frame_bgr = capture.read()
            if not success:
                break

            cropped_frame_bgr = apply_side_crop(
                stacked_frame_bgr,
                crop_info["crop_left"],
                crop_info["crop_right"],
            )
            front_frame_bgr, rear_frame_bgr = split_front_rear_frame(cropped_frame_bgr)
            front_is_uniform = is_nearly_uniform_frame(front_frame_bgr)
            rear_is_uniform = is_nearly_uniform_frame(rear_frame_bgr)

            if front_start_open and front_is_uniform:
                front_uniform_start += 1
            else:
                front_start_open = False

            if rear_start_open and rear_is_uniform:
                rear_uniform_start += 1
            else:
                rear_start_open = False

            front_uniform_end = front_uniform_end + 1 if front_is_uniform else 0
            rear_uniform_end = rear_uniform_end + 1 if rear_is_uniform else 0
            total_frame_pairs += 1
    finally:
        capture.release()

    removed_from_start = max(front_uniform_start, rear_uniform_start)
    removed_from_end = max(front_uniform_end, rear_uniform_end)
    remaining_frame_pairs = max(total_frame_pairs - removed_from_start - removed_from_end, 0)

    return {
        "input_frame_pairs": int(total_frame_pairs),
        "front_uniform_frames_at_start": int(front_uniform_start),
        "rear_uniform_frames_at_start": int(rear_uniform_start),
        "front_uniform_frames_at_end": int(front_uniform_end),
        "rear_uniform_frames_at_end": int(rear_uniform_end),
        "removed_from_start": int(removed_from_start),
        "removed_from_end": int(removed_from_end),
        "remaining_frame_pairs": int(remaining_frame_pairs),
    }


def create_mp4_writer(output_path: Path, fps: float, frame_shape: tuple[int, ...]) -> cv2.VideoWriter:
    output_path.parent.mkdir(parents=True, exist_ok=True)
    height, width = frame_shape[:2]
    fourcc = cv2.VideoWriter_fourcc(*MOVIE_PREPROCESS_FOURCC)
    writer = cv2.VideoWriter(str(output_path), fourcc, fps, (int(width), int(height)))
    if not writer.isOpened():
        raise RuntimeError(f"Failed to open video writer: {output_path}")
    return writer


def build_movie_preprocess_output_paths(
    video_path: Path,
    category: str,
    output_base_dir: Path = MOVIE_PREPROCESS_OUTPUT_DIR,
) -> tuple[Path, Path]:
    front_output_path = output_base_dir / category / "front" / f"{video_path.stem}_front.mp4"
    rear_output_path = output_base_dir / category / "rear" / f"{video_path.stem}_rear.mp4"
    return front_output_path, rear_output_path


def preprocess_movie_to_front_rear(
    video_path: Path,
    category: str,
    output_base_dir: Path = MOVIE_PREPROCESS_OUTPUT_DIR,
    overwrite: bool = MOVIE_PREPROCESS_OVERWRITE,
) -> dict[str, Any]:
    metadata = read_video_metadata(video_path)
    native_fps = metadata["fps"] if metadata["fps"] > 0 else 30.0
    front_output_path, rear_output_path = build_movie_preprocess_output_paths(
        video_path,
        category,
        output_base_dir,
    )

    result: dict[str, Any] = {
        "category": category,
        "input_video_path": str(video_path),
        "front_output_path": str(front_output_path),
        "rear_output_path": str(rear_output_path),
        "input_frame_count": int(metadata["frame_count"]),
        "input_fps": float(metadata["fps"]),
        "input_width": int(metadata["width"]),
        "input_height": int(metadata["height"]),
        "input_duration_sec": float(metadata["duration_sec"]),
    }

    if not overwrite and front_output_path.exists() and rear_output_path.exists():
        result.update({"status": "skipped_existing_outputs"})
        return result

    sampled_frames = iter_sampled_frames(video_path)
    result["sampled_frame_count"] = int(len(sampled_frames))
    if not sampled_frames:
        result.update({"status": "skipped_no_sampled_frames"})
        return result

    _, crop_info = preprocess_stacked_frames_remove_letterbox(sampled_frames)
    trim_info = estimate_full_video_uniform_trim(video_path, crop_info)
    result.update({
        "crop_left": int(crop_info["crop_left"]),
        "crop_right": int(crop_info["crop_right"]),
        "cropped_width": int(crop_info["cropped_width"]),
        "removed_left_px": int(crop_info["removed_left_px"]),
        "removed_right_px": int(crop_info["removed_right_px"]),
    })
    result.update(trim_info)

    if trim_info["remaining_frame_pairs"] <= 0:
        result.update({"status": "skipped_no_frames_after_trim", "written_frame_pairs": 0})
        return result

    capture = cv2.VideoCapture(str(video_path))
    if not capture.isOpened():
        raise FileNotFoundError(f"Failed to open video: {video_path}")

    front_writer = None
    rear_writer = None
    frame_pair_index = 0
    written_frame_pairs = 0
    start_index = trim_info["removed_from_start"]
    end_index = trim_info["input_frame_pairs"] - trim_info["removed_from_end"]

    try:
        while capture.isOpened():
            success, stacked_frame_bgr = capture.read()
            if not success:
                break

            should_write = start_index <= frame_pair_index < end_index
            if should_write:
                cropped_frame_bgr = apply_side_crop(
                    stacked_frame_bgr,
                    crop_info["crop_left"],
                    crop_info["crop_right"],
                )
                front_frame_bgr, rear_frame_bgr = split_front_rear_frame(cropped_frame_bgr)

                if front_writer is None or rear_writer is None:
                    front_writer = create_mp4_writer(front_output_path, native_fps, front_frame_bgr.shape)
                    rear_writer = create_mp4_writer(rear_output_path, native_fps, rear_frame_bgr.shape)
                    result.update({
                        "output_width": int(front_frame_bgr.shape[1]),
                        "output_height": int(front_frame_bgr.shape[0]),
                    })

                front_writer.write(front_frame_bgr)
                rear_writer.write(rear_frame_bgr)
                written_frame_pairs += 1

            frame_pair_index += 1
    finally:
        capture.release()
        if front_writer is not None:
            front_writer.release()
        if rear_writer is not None:
            rear_writer.release()

    result.update({
        "status": "written",
        "written_frame_pairs": int(written_frame_pairs),
    })
    return result


def run_movie_preprocess_for_dataset(
    output_base_dir: Path = MOVIE_PREPROCESS_OUTPUT_DIR,
    manifest_path: Path = MOVIE_PREPROCESS_MANIFEST_PATH,
    overwrite: bool = MOVIE_PREPROCESS_OVERWRITE,
) -> pd.DataFrame:
    targets = find_movie_preprocess_targets()
    print("movie preprocess target count:", len(targets))
    rows: list[dict[str, Any]] = []

    for target_index, target in enumerate(targets, start=1):
        category = target["category"]
        video_path = target["video_path"]
        print(f"[{target_index}/{len(targets)}] {category}: {video_path.name}")
        try:
            result = preprocess_movie_to_front_rear(
                video_path=video_path,
                category=category,
                output_base_dir=output_base_dir,
                overwrite=overwrite,
            )
        except Exception as error:
            front_output_path, rear_output_path = build_movie_preprocess_output_paths(
                video_path,
                category,
                output_base_dir,
            )
            result = {
                "category": category,
                "input_video_path": str(video_path),
                "front_output_path": str(front_output_path),
                "rear_output_path": str(rear_output_path),
                "status": "error",
                "error": repr(error),
            }
        rows.append(result)
        print("  status:", result.get("status"), "written_frame_pairs:", result.get("written_frame_pairs"))

    manifest_df = pd.DataFrame(rows)
    manifest_path.parent.mkdir(parents=True, exist_ok=True)
    manifest_df.to_csv(manifest_path, index=False)
    print("movie preprocess manifest saved to:", manifest_path)
    if "status" in manifest_df:
        print("status counts:")
        print(manifest_df["status"].value_counts(dropna=False).to_dict())
    return manifest_df


if VIDEO_PROCESSING_ENABLED:
    movie_preprocess_manifest_df = run_movie_preprocess_for_dataset()
else:
    print("Skip movie preprocessing because VIDEO_PROCESSING_ENABLED is False.")
